# Launch a run

**This notebook defines the config.** It is the one place a run is written, and everything downstream derives from what the config cell says: the ETL reads its `encoder` and `sources` blocks to build a DAG, the trainer reads the rest.

A run is one folder and one file — `runs/{run_id}/config.json`. The rest of the folder is derived: the request the ETL resolved, the bins it gathered, the checkpoints the worker wrote. So the job here is to define that file, check it, put it on the Volume, and print the commands that act on it.

Nothing here spends money. It uploads a few hundred bytes; the commands it prints are what start containers, and you run those yourself.

The same bytes are also written to `notebooks/pipeline/runs/local/config.json`, which is what `etl_train_pipeline.ipynb` builds against when you want to run the DAG on your laptop.

In [33]:
import io
import json
import sys

import torch

from config import PROJECT_ROOT, get_git_commit
from transformer import TransformerLM
from transformer.optimizer import lr_cosine_schedule
from transformer.util import resolve_config, serialize_config

# The repo root is already on sys.path -- transformer.pth, from the editable install
# -- which is what makes `config` and `transformer` importable from any directory.
# These two are not: launcher/ and notebooks/pipeline/ hold plain modules rather than
# installed packages, so they have to be named.
PIPELINE = PROJECT_ROOT / "notebooks" / "pipeline"
sys.path.insert(0, str(PROJECT_ROOT / "launcher"))  # app.py, jobs.py
sys.path.insert(0, str(PIPELINE))  # etl.py, beside the Snakefile that shares it

import etl
import modal

# From app.py rather than as literals, so this cannot name a different volume, app
# or GPU than the deployment it launches into. Importing it also proves it imports.
from app import APP_NAME, GPU, VOLUME_NAME
from jobs import mint_run_id

volume = modal.Volume.from_name(VOLUME_NAME, create_if_missing=True)
print(f"app     {APP_NAME}")
print(f"volume  {VOLUME_NAME}")
print(f"gpu     {GPU}")

app     training-launcher
volume  test-volume
gpu     T4


## The run

Live class and function references, not strings: `serialize_config` encodes them as importable `module.qualname` on the way to config.json, and `resolve_config` reverses it in the training container. Writing them live means the interpreter checks this cell as you edit it.

`VOCAB_SIZE` is one name used twice on purpose — the embedding table is sized from `model_params` and the vocabulary is fit from `encoder.params`, and those two disagreeing is silent until a worker already holds a lease.

Editing the `encoder` block changes its hash, so it builds a new encoder beside the old one rather than replacing it: cheap to say, not cheap to run.

In [41]:
NAME = "run2"  # the readable half of the run_id
VOCAB_SIZE = 2000  # sizes the embedding table *and* fits the encoder

# Hoisted out of the dict below so the tags can name them, and so each is still
# written exactly once.
#
# Sources by uid in the catalog (notebooks/pipeline/sources.yaml), not paths -- where
# a bin lands depends on the encoder. Overlap between the two sides, or a valid source
# in the encoder's fit_sources, is rejected before anything is built.
SOURCES = {
    "train_sources": ["odyssey"],
    "valid_sources": ["montecristo", "romeojuliet", "mobydick"],
}

# The encoder as a definition, not a name to look up: this block is hashed into the
# encoder_uid that names its directory. `kind` picks an implementation from
# encoders.py and `params` is passed to it as keyword arguments.
ENCODER = {
    "kind": "bpe",
    "params": {
        "vocab_size": VOCAB_SIZE,
        "special_tokens": ["<|endoftext|>", "<|begin|>", "<|end|>"],
    },
    "fit_sources": ["odyssey"],
}

# W&B tags, and why they are worth the trouble: `sources` does land in the run's
# config, but a list-valued config field shows up in the runs table as a string that
# looks like a Python list, which is not something you can filter on. Tags are.
#
# Tags are a set of strings with no keys, so which side a source was on has to be
# carried in the string itself -- and a source moves sides between runs, so that is
# the part worth keeping. Run tags accept any character and are only bounded at 64
# (the stricter alphanumeric rule is for *artifact* tags), so prefixes are safe.
#
# `enc:` makes "every run that shared this encoder" one click, which is the query the
# uid hashing exists to support. Naming the uid here cannot change it: encoder_uid
# hashes kind/params/fit_sources and nothing else, and metadata is not part of it.
TAGS = (
    [f"run:{NAME}"]
    + [f"train:{uid}" for uid in SOURCES["train_sources"]]
    + [f"valid:{uid}" for uid in SOURCES["valid_sources"]]
    + [f"enc:{etl.encoder_uid(ENCODER)}"]
)

config = {
    "description": NAME,
    "seed": 0,
    "model_class": TransformerLM,
    "model_params": {
        "vocab_size": VOCAB_SIZE,
        "context_length": 64,
        "num_layers": 1,
        "d_model": 64,
        "d_ff": 128,
        "num_heads": 1,
        "rope_theta": 10000,
        "device": "cuda",  # has to agree with app.GPU; the preflight checks it
        "dtype": None,
    },
    "optimizer_class": torch.optim.AdamW,
    "optimizer_params": {
        "lr": 0.001,
        "betas": (0.9, 0.999),
        "weight_decay": 0.1,
        "eps": 1e-8,
    },
    "lr_schedule_fn": lr_cosine_schedule,
    "lr_schedule_params": {
        "max_learning_rate": 0.001,
        "min_learning_rate": 0.0001,
        "warmup_iters": 30,
        "cosine_cycle_iters": 1000,
    },
    # No train_path/valid_path: a run trains on its own folder's train.bin and
    # valid.bin, which the ETL gathers from the sources below. Naming paths here
    # would be a second way of saying what `sources` already says.
    "training": {
        "total_steps": 5000,
        "batch_size": 32,
        "val_every": 10,
        "save_every": 1000,
        "gpu_check_every": 50,
        "max_norm": 1.0,  # gradient-clipping threshold, passed to clip_grad_norm_
    },
    "sources": SOURCES,
    "encoder": ENCODER,
    # Passed straight to wandb.init: project/name/notes/tags become native W&B
    # fields, anything else here becomes a flat config column. Delete this whole
    # block to turn W&B off -- WandbRun keys off its presence.
    #
    # Tags are set once, at run creation, and are sticky: on every resume wandb
    # re-attaches to the existing run and the server's tags overwrite whatever init
    # passes. Fine here -- config.json is write-once, so the sources cannot change
    # mid-run -- but editing TAGS and resuming changes nothing in W&B.
    "metadata": {
        "git_commit": get_git_commit(True),
        "project": "llm-pretraining",
        "name": NAME,
        "notes": "",
        "tags": TAGS,
    },
}

# What actually ships: the JSON-safe form. Everything below checks this, not the
# dict above, so what is checked is what lands on the Volume.
payload = serialize_config(config)
print(json.dumps(payload, indent=2))


{
  "description": "run2",
  "seed": 0,
  "model_class": "transformer.model.TransformerLM",
  "model_params": {
    "vocab_size": 2000,
    "context_length": 64,
    "num_layers": 1,
    "d_model": 64,
    "d_ff": 128,
    "num_heads": 1,
    "rope_theta": 10000,
    "device": "cuda",
    "dtype": null
  },
  "optimizer_class": "torch.optim.adamw.AdamW",
  "optimizer_params": {
    "lr": 0.001,
    "betas": [
      0.9,
      0.999
    ],
    "weight_decay": 0.1,
    "eps": 1e-08
  },
  "lr_schedule_fn": "transformer.optimizer.lr_cosine_schedule",
  "lr_schedule_params": {
    "max_learning_rate": 0.001,
    "min_learning_rate": 0.0001,
    "warmup_iters": 30,
    "cosine_cycle_iters": 1000
  },
  "training": {
    "total_steps": 5000,
    "batch_size": 32,
    "val_every": 10,
    "save_every": 1000,
    "gpu_check_every": 50,
    "max_norm": 1.0
  },
  "sources": {
    "train_sources": [
      "odyssey"
    ],
    "valid_sources": [
      "montecristo",
      "romeojuliet",
      "mo

## Preflight

Every check that can be made from here, made from here. Each of these otherwise costs a container start to discover, and two of them only surface once a worker is already holding a lease.

In [42]:
# What the ETL will ask the DAG for: unknown source uids, train/valid overlap, a
# validation source leaked into the encoder's fit set. The same function the
# Snakefile calls, so a config that passes here cannot fail there.
spec = etl.spec(payload)
etl.validate(spec, etl.read_catalog(PIPELINE))

# Guards a hand-edit that breaks what VOCAB_SIZE ties together above.
assert payload["model_params"]["vocab_size"] == payload["encoder"]["params"]["vocab_size"], (
    f"model vocab_size {payload['model_params']['vocab_size']} != "
    f"encoder vocab_size {payload['encoder']['params']['vocab_size']}"
)

# `work` gets a GPU or it does not, and torch raises at model construction if the
# config disagrees. app.GPU and this key have to move together.
device = payload["model_params"]["device"]
assert (device == "cuda") == (GPU is not None), (
    f"config device is {device!r} but app.GPU is {GPU!r} -- set both, or neither, for a GPU"
)

# The keys Train reads, by the names it reads them under.
for key in ("total_steps", "save_every", "val_every", "batch_size", "max_norm"):
    assert key in payload["training"], f"training.{key} missing -- Train aborts on it"

# W&B's own rules for run tags: str, 1 to 64 characters. It enforces them inside
# wandb.init, which a worker reaches only after the ETL has run and it is already
# holding the lease -- so an unusable tag would cost a build and a container to find.
tags = payload.get("metadata", {}).get("tags", [])
bad = [t for t in tags if not isinstance(t, str) or not 1 <= len(t) <= 64]
assert not bad, f"W&B rejects these tags (need str, 1-64 chars): {bad}"

# The round trip the training container performs: strings back to real objects.
resolved = resolve_config(payload)
for key in ("model_class", "optimizer_class", "lr_schedule_fn"):
    assert callable(resolved[key]), f"{key} did not resolve: {payload[key]!r}"

print(f"encoder_uid   {spec['encoder']['encoder_uid']}")
print(f"train         {spec['sources']['train_sources']}")
print(f"valid         {spec['sources']['valid_sources']}")
print(f"steps         {payload['training']['total_steps']} every {payload['training']['save_every']}")
print(f"device        {device}   (app.GPU {GPU})")
print(f"tags          {tags}")
print("\npreflight passed")


encoder_uid   bpe-611c2460
train         ['odyssey']
valid         ['montecristo', 'romeojuliet', 'mobydick']
steps         5000 every 1000
device        cuda   (app.GPU T4)
tags          ['run:run2', 'train:odyssey', 'valid:montecristo', 'valid:romeojuliet', 'valid:mobydick', 'enc:bpe-611c2460']

preflight passed


## Push it

A fresh `run_id` from `jobs.mint_run_id` — the same function `launch` uses, so an id minted here is indistinguishable from one it minted itself.

`force=False`: a run's config.json is written once and every later attempt runs under it, so this refuses to overwrite rather than silently redefining a run that may already have checkpoints. Re-run the cell for a new id instead.

The local copy under `runs/local/` is the same bytes, for driving the DAG on your laptop from `etl_train_pipeline.ipynb`. It is not uploaded and nothing on Modal reads it.

In [43]:
local = etl.write_config(payload, etl.run_dir(PIPELINE, "local"))
print(f"local   {local.relative_to(PROJECT_ROOT)}")

run_id = mint_run_id(NAME)
remote = f"/runs/{run_id}/{etl.CONFIG_FILE}"
blob = json.dumps(payload, indent=2).encode()

with volume.batch_upload(force=False) as batch:
    batch.put_file(io.BytesIO(blob), remote)

print(f"volume  {VOLUME_NAME}:{remote}  ({len(blob):,} bytes)")
for entry in volume.listdir(f"runs/{run_id}"):
    print(f"        {entry.path}  {entry.size:,} bytes")

local   notebooks/pipeline/runs/local/config.json
volume  test-volume:/runs/20260813T050943_run2/config.json  (1,572 bytes)
        runs/20260813T050943_run2/config.json  1,572 bytes


## Run it

Two steps, in this order, because they are two different kinds of spend and the first is the one that can take a while.

**`etl`** runs the snakemake workflow against the Volume: it downloads whatever sources this config names, fits the encoder if no run has fit that exact definition before, encodes each source, then joins this run's bins into `train.bin`/`valid.bin` in its folder. Anything already built is skipped — every target is addressed by uid, so a config asking for a known encoder comes back in seconds.

**`launch`** starts the worker: one container, one lease, training to `total_steps`. Safe to call repeatedly — it is the same call whether you are starting a run, resuming one after a crash, or asking about a finished one, and its `reason` tells you which happened.

Both reach the **deployed** app, which has to exist first:

```
modal deploy launcher/app.py
```

That is not optional bookkeeping. `launch`'s `max_containers=1` is the only mutual exclusion in the system and it holds only within one deployment, so a locally-constructed copy would run in its own container pool and serialize against nothing.

In [ ]:
# Handles on the deployed functions.
#
# from_name is lazy -- it does not talk to Modal until the call -- so hydrate() here
# turns "nothing is deployed" into one line now, instead of a NotFoundError traceback
# out of the middle of .remote() later.
def deployed(name: str) -> modal.Function:
    fn = modal.Function.from_name(APP_NAME, name)
    try:
        fn.hydrate()
    except modal.exception.NotFoundError:
        raise RuntimeError(
            f"no {name!r} on a deployed {APP_NAME!r} -- run `modal deploy launcher/app.py` first"
        ) from None
    return fn


# `etl` is the pipeline module in this notebook, so the Modal function has to be bound
# to another name or the import at the top is shadowed.
etl_fn = deployed("etl")
launch_fn = deployed("launch")
print(f"reachable on {APP_NAME}: etl, launch")


In [ ]:
# 1. Build this run's data, on its own -- the notebook equivalent of
#    `modal run launcher/app.py::prepare --run-id ...`
#
# Blocking, and only the return value comes back here: snakemake's own output goes to
# the container log, so watch `modal app logs` if you want to see the DAG run. A
# non-zero exit surfaces as a RuntimeError from this cell.
#
# Idempotent -- this is also how you rebuild after deleting an artifact.
print(etl_fn.remote(run_id))

# What it left in the run's folder: run.yaml (the resolved request), encoder.json
# (what encoded these tokens), and the two bins the trainer memory-maps.
for entry in sorted(volume.listdir(f"runs/{run_id}"), key=lambda e: e.path):
    print(f"  {entry.size:>12,}  {entry.path}")


In [ ]:
# 2. Train it -- the notebook equivalent of
#    `modal run launcher/app.py --run-id ...`
#
# Passing an existing run_id is the resume path, which for a folder with no checkpoints
# means starting it. That path deliberately does not run the ETL, so the cell above has
# to have succeeded or the worker aborts on a missing train.bin.
#
# Returns immediately: `launch` spawns the worker and grants the lease, it does not
# wait for training. Re-run it as often as you like -- `reason` distinguishes a fresh
# spawn from "already running", "already complete", and an unreadable folder.
print(launch_fn.remote(run_id=run_id))


In [ ]:
print(f"""
# 1. build the data this config asks for (skips anything already built)
modal run launcher/app.py::prepare --run-id {run_id}

# 2. train it -- also the command to resume, as often as you like
modal run launcher/app.py --run-id {run_id}

# watch it
modal app logs {APP_NAME}
modal volume ls {VOLUME_NAME} runs/{run_id}
""")

In [ ]:
APP_NAME, NAME

In [ ]:
modal.Function.from_name(APP_NAME, "launch").remote(name=NAME, config=payload)

### The one-call alternative

`launch` can do all of it in a single call — minting the `run_id`, writing config.json itself, running the ETL, spawning the worker — with no upload and no commands:

```python
modal.Function.from_name(APP_NAME, "launch").remote(name=NAME, config=payload)
```

One step instead of three, at the cost of starting containers from a notebook cell and holding the launcher's single container for the whole build. This notebook is the version where you see the config, then decide.

### If the upload cell fails

The same push from a terminal, using the local copy the push cell wrote:

```
modal volume put <volume> notebooks/pipeline/runs/local/config.json /runs/<run_id>/config.json
```